# Benchmarks

## Initialize

In [2]:
import os
import math
import pathlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
from tqdm.auto import tqdm
from IPython.display import clear_output

import warnings
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

In [ ]:
import ray
ray.shutdown()

In [3]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

experiment = 230425
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [4]:
import ray
ray.init(num_cpus=24, include_dashboard=False)#, dashboard_port=24762, dashboard_host="0.0.0.0", include_dashboard=True)#, webui_url="0.0.0.0"))

2023-04-26 11:26:57,564	INFO worker.py:1529 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8266 


Python version:,3.10.9
Ray version:,2.2.0
Dashboard:,http://127.0.0.1:8266


In [6]:
endpoints_md = pd.read_csv(f"{experiment_path}/endpoints.csv")
endpoints = sorted(endpoints_md.endpoint.to_list())

In [7]:
in_path = pathlib.Path(f"{experiment_path}/loghs")
in_path.mkdir(parents=True, exist_ok=True)

In [8]:
models = models = [f.name for f in in_path.iterdir() if f.is_dir() and "ipynb_checkpoints" not in str(f)]
partitions = [i for i in range(22)] #[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

In [9]:
from sklearn.preprocessing import StandardScaler
import pickle
import zstandard

def read_data(fp_in, split):
    temp = pd.read_feather(f"{fp_in}/{split}.feather").set_index("eid")
    return temp   
    
def save_pickle(data, data_path):
    with open(data_path, "wb") as fh:
        cctx = zstandard.ZstdCompressor()
        with cctx.stream_writer(fh) as compressor:
            compressor.write(pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL))
    
def read_predictions(model, partition, split):

    fp_in = f"{in_path}/{model}/{partition}"
    
    if pathlib.Path(fp_in).is_dir(): 
        temp = read_data(fp_in, split)
        
    return temp

In [10]:
for partition in partitions:
    for split in ["train", "valid", "test"]:
        try: 
            temp = read_predictions('Identity(Records)+MLP', partition, split)
            print(partition, split, (temp.isna().sum() > 0).sum())
        except UnboundLocalError:
            print(partition, split, "not available")

0 train 0
0 valid 0
0 test 0
1 train 0
1 valid 0
1 test 0
2 train 0
2 valid 0
2 test 0
3 train 0
3 valid 0
3 test 0
4 train 0
4 valid 0
4 test 0
5 train 0
5 valid 0
5 test 0
6 train 0
6 valid 0
6 test 0
7 train 0
7 valid 0
7 test 0
8 train 0
8 valid 0
8 test 0
9 train 0
9 valid 0
9 test 0
10 train 0
10 valid 0
10 test 0
11 train 0
11 valid 0
11 test 0
12 train 0
12 valid 0
12 test 0
13 train 0
13 valid 0
13 test 0
14 train 0
14 valid 0
14 test 0
15 train 0
15 valid 0
15 test 0
16 train 0
16 valid 0
16 test 0
17 train 0
17 valid 0
17 test 0
18 train 0
18 valid 0
18 test 0
19 train 0
19 valid 0
19 test 0
20 train 0
20 valid 0
20 test 0
21 train 0
21 valid 0
21 test 0


(raylet) [2023-04-26 16:33:57,734 E 2194151 2194151] (raylet) node_manager.cc:3097: 1 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 4f92fb4cda42ae54e7dba9f6baaf6a935ae8a3b85be7316a5c6068ed, IP: 10.32.105.119) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 10.32.105.119`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.
(raylet) [2023-04-26 16:37:08,284 E 2194151 2194151] (raylet) node_manager.cc:3097: 1 Workers (tasks / actors) 